# 분류? 까이꺼 함 해보죠.
- 사실 머신러닝은 처음이지만... 예. 해봅시다... 그 전에 데이터 갖고오기부터 해야겠지만.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import kdeplot

from sklearn.impute import SimpleImputer # 임퓨퉈퉈퉈
from sklearn.impute import KNNImputer # 같은일 하는 친구입니다
from sklearn.preprocessing import StandardScaler, OneHotEncoder # 마! 서탠다더!
from sklearn.compose import ColumnTransformer # 누... 누구세요?
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 모델
from sklearn.ensemble import RandomForestClassifier # Random Forest
from sklearn.svm import SVC # 서포트 벡터 머쉬이이인
from xgboost import XGBClassifier # XGBoost
import lightgbm as lgb # LightGBM

# 성적표
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="icefire", style="darkgrid", font_scale=1)
sns.color_palette("tab10", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Umdot 12'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("parulpandey/palmer-archipelago-antarctica-penguin-data")

print("Path to dataset files:", path)

In [ ]:
penguin = pd.read_csv(f'{path}/penguins_lter.csv')

# 파일 정보 확인

## df.info()

In [ ]:
penguin.info()
# 아니! 결측값! 먼데!

## df.describe()

In [ ]:
penguin.describe()

In [ ]:
penguin.describe(include='O')

## df.isna().sum()

In [ ]:
penguin.isna().sum()

## df.head()

In [ ]:
penguin.head()

## df.columns

In [ ]:
penguin.columns

# 전처리
- 사실 전처리... 지금까지 EDA 코드를 보면 그냥 결측값 때우고 범주화하고 들어갔죠? 근데 머신러닝에서는 아무 칼럼이나 덮어놓고 학습에 썼다간 성능이 제대로 안 나올 수도 있습니다. 무슨 말인지 아시죠? 그래서 결측값 땜질하고 인코딩 하기 전에 필요한 칼럼만 먼저 추릴거예요.

## 필요한 칼럼 추리기
- 위에 헤드를 다시 가져와서 내용물이 뭐뭐 있는지 봅시다. 

In [ ]:
penguin.head()

- 날릴 칼럼: studyName, Sample Number, Individual ID, Commments, Region(단일이라...), Clutch Completion(우리 종 나눌거라 이건 필요가 없음), Date Egg

In [ ]:
# Copy
penguin_drop = penguin.copy()

# 하고 날려날려 칼럼
penguin_drop.drop(['studyName','Sample Number','Region','Individual ID','Clutch Completion', 'Comments', 'Date Egg', 'Stage'], axis=1, inplace=True)
penguin_drop.head()

## 결측값 때우기
- 유형 잘 보고 때워야 합니다. 일단 임퓨터 커몬.

In [ ]:
penguin_drop.isna().sum()

### Culmen Length (mm)
- 이거 분포 무슨 코끼리를 삼킨 보아뱀이던데

In [ ]:
sns.kdeplot(penguin_drop['Culmen Length (mm)'])

In [ ]:
na_idx = penguin_drop.query('`Culmen Length (mm)`.isna()').index
for i in na_idx:
    print(penguin_drop.loc[i])

- 이거 일단 몸무게까지는 같은 방식으로 떄웁시다. 2개 비어있는데 그 비어있는게 다 얘네들임.

In [ ]:
num_cols = ['Culmen Length (mm)', 'Culmen Depth (mm)', 'Flipper Length (mm)', 'Body Mass (g)'] # 일단 떄울 칼럼

imputer = KNNImputer(n_neighbors=5) # 얘는 그 이웃 참고해서 때워주는 친구입니다
penguin_drop[num_cols] = imputer.fit_transform(penguin_drop[num_cols]) # 때-움

In [ ]:
penguin_drop.isna().sum()

### 성별

In [ ]:
na_idx = penguin_drop.query('Sex.isna()').index
for i in na_idx:
    print(penguin_drop.loc[i])

- 일단 범주형이라 최빈값으로 가는 게 맞는 것 같기도...
- 그리고 저기 . 있어서 그것도 결측값으로 바꾸고 가겠습니다. 

In [ ]:
# . 대치합니닷
penguin_drop['Sex'] = penguin_drop['Sex'].replace('.', np.nan)

penguin_drop['Sex'].unique()

In [ ]:
imputer = SimpleImputer(strategy='most_frequent')
penguin_drop[['Sex']] = imputer.fit_transform(penguin_drop[['Sex']]) # 때-움

In [ ]:
penguin_drop.isna().sum()

### 동위원소
- ...뭐지 이건?

In [ ]:
na_idx = penguin_drop.query('`Delta 15 N (o/oo)`.isna()').index
for i in na_idx:
    print(penguin_drop.loc[i])

In [ ]:
num_cols = ['Delta 15 N (o/oo)', 'Delta 13 C (o/oo)'] # 일단 떄울 칼럼

imputer = KNNImputer(n_neighbors=5) # 얘는 그 이웃 참고해서 때워주는 친구입니다
penguin_drop[num_cols] = imputer.fit_transform(penguin_drop[num_cols]) # 때-움

In [ ]:
penguin_drop.isna().sum()

- 깔-끔

## 마! 서케일러!
- 이걸 왜 하냐고요? 수치형 숫자가 다 달라서요.

In [ ]:
scaler = StandardScaler() # 스케일러가 요기잉눼?

num_features = ['Culmen Length (mm)', 'Culmen Depth (mm)', 'Flipper Length (mm)',
                'Body Mass (g)', 'Delta 15 N (o/oo)', 'Delta 13 C (o/oo)'] # 스케일러
cat_features = ['Island', 'Sex'] # 인코더

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])

X_processed = preprocessor.fit_transform(penguin_drop)

In [ ]:
X_processed

In [ ]:
# 원-핫 인코딩된 피처 이름 추출
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features)
# 전체 피처 이름 합치기
all_feature_names = num_features + list(cat_names)

print(all_feature_names)

# 학습 히위고 디비고 렛츄고
- 근데 여기 있는 데이터셋을 다 학습하는 데 쓰는 게 아니라 일부를 테스트용으로 쓸겁니다.
- 그래서 뭐 하냐고요? 일단 째야 뭘 하죠.

In [ ]:
y = penguin_drop['Species'] # 문제지
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size = 0.2, random_state=42, stratify=y) # 기본값 8:2

print(f'전체 데이터의 수 : {len(X_processed)}')
print(f'학습 데이터의 수 : {len(X_train)}')
print(f'테스트 테이터의 수 : {len(X_test)}')

## 랜덤포레스트

In [ ]:
forest = RandomForestClassifier(random_state=42) # 얘는 근데 숲이랑 뭔 상관이 있길래 이름이 랜덤포리스트인겨
forest.fit(X_train, y_train) # 학습
forest_pred = forest.predict(X_test)

- 이게 다냐고요? 예.

In [ ]:
# 얼마나 맞췄는지 점수(%) 확인
print(f"정확도: {accuracy_score(y_test, forest_pred):.3f}")

# 종별로 얼마나 잘 분류했는지 상세 리포트
print(classification_report(y_test, forest_pred))

In [ ]:
# 혼동 행렬 시각화
cm = confusion_matrix(y_test, forest_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=forest.classes_)
disp.plot(cmap='Blues')
plt.show()

In [ ]:
# 중요도 시각화
importances = pd.Series(forest.feature_importances_, index=all_feature_names)
importances.sort_values().plot(kind='barh')
plt.show()

## SVM(서포트 벡터 머신)

In [ ]:
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)

In [ ]:
# 1. 성적표 (Classification Report)
print("--- SVM 분류 성적표 ---")
print(classification_report(y_test, svm_predictions))

# 2. 오답 분석 (Confusion Matrix)
cm = confusion_matrix(y_test, svm_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=svm_model.classes_)
disp.plot(cmap='viridis')
plt.title("SVM Confusion Matrix")
plt.show()

## XGBoost
- 이거 ADsP 할 때 들어본 것도 같고...

In [ ]:
# 1. 변환기 생성
le = LabelEncoder()

# 2. 정답지(y)를 숫자로 변환
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# 다중 분류이므로 objective를 'multi:softmax'로 설정하는 게 정석입니다
xgb_model = XGBClassifier(objective='multi:softmax', n_estimators=50, random_state=42)
xgb_model.fit(X_train, y_train_encoded)

# 결과 확인
print(classification_report(y_test, le.inverse_transform(xgb_model.predict(X_test))))

- ?? 랜덤포레스트랑 비슷한데 성능?

## LightGBM
- lightbgm이라고 쳐놓고 아나콘다 왜 못찾지? 이러고 있었음...ㅋㅋㅋㅋ

In [ ]:
lgbm_model = lgb.LGBMClassifier(n_estimators=10, random_state=42, verbose=-1)
lgbm_model.fit(X_train, y_train_encoded)

# 1. 모델이 예측한 값(숫자)을 받아옵니다
lgb_pred_encoded = lgbm_model.predict(X_test)

# 2. 숫자를 다시 원래 펭귄 이름(문자열)으로 돌립니다
# 여기서 le(LabelEncoder)가 아까 학습(fit)된 상태여야 합니다
lgb_pred = le.inverse_transform(lgb_pred_encoded)

In [ ]:
print("--- LightGBM 분류 성적표 ---")
print(classification_report(y_test, lgb_pred))

# 3. 마지막 혼동 행렬 시각화
cm = confusion_matrix(y_test, lgb_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=lgbm_model.classes_)
disp.plot(cmap='Greens')
plt.title("LightGBM Confusion Matrix")
plt.show()

- 빨리 GG친 것 치곤 얘가 제일 꼴찌입니다...
- 근데 그럴수밖에 없는게, XGBoost나 LightGBM은 스케일이 큰 애들입니다. 걔네 입장에서 이 데이터는 약간 어떤 느낌이냐면 일론 머스크가 10달러 쓰는 느낌이죠. 